# PromptAD — paper-faithful checkpoint training on Kaggle

This notebook retrains the official CVPR 2024 [PromptAD](https://github.com/FuNz-0/PromptAD) implementation at pinned commit `0f86ce0dc1ed59007d51348d8d566aed31360cf9`, using the [paper and supplementary training setup](https://arxiv.org/abs/2404.05231).

PromptAD's released code does **not** produce one global model. It trains a separate class-specific checkpoint for every dataset, shot count, and task. A full MVTec AD + VisA, 1/2/4-shot reproduction therefore contains `27 classes × 3 shots × 2 tasks = 162` checkpoints. CLS and SEG run concurrently on separate T4 GPUs; use `MAX_JOBS_THIS_SESSION` and `RESUME_RESULT_ROOT` to split the exact suite over Kaggle sessions.

Important fidelity notes:

- The official 100-epoch scripts evaluate the test set after every epoch and retain the epoch with the best test AUROC. This notebook deliberately preserves that published-code behavior.
- The saved official checkpoint contains the two normal visual galleries and the learned two-class text-feature gallery (`feature_gallery1`, `feature_gallery2`, and `text_features`), not the frozen LAION-400M CLIP backbone or raw prompt-token parameters.
- `train_cls.py` officially uses 3 normal prompt prototypes, while `train_seg.py` uses 1. This notebook follows the pinned official scripts. The supplementary prose says 1, but the paper's visualization discusses 3 and the official CLS default is 3.
- Internet must be enabled so Kaggle can clone the source and download the LAION-400M ViT-B/16+ weights on the first run.

In [ ]:
import hashlib
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

print("====== STEP 1: INSTALLING RUNTIME AND PINNING OFFICIAL SOURCE ======")

# Kaggle already supplies CUDA-enabled torch/torchvision. Do not replace them.
# PromptAD vendors OpenCLIP 2.16 source but imports one helper from the package.
REQUIRED_PACKAGES = [
    "open_clip_torch>=2.16,<3",
    "tqdm>=4.64",
    "ftfy>=6.1",
    "regex>=2023.0",
    "loguru>=0.7",
    "timm>=0.9,<2",
    "scipy>=1.9",
    "scikit-image>=0.20",
    "scikit-learn>=1.2",
    "pandas>=1.5",
    "seaborn>=0.12",
    "opencv-python-headless>=4.8",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *REQUIRED_PACKAGES],
    check=True,
)

import cv2
import numpy as np
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "PromptAD's official implementation is fp16-only and requires CUDA. "
        "In Kaggle, select Settings -> Accelerator -> GPU."
    )

PROMPTAD_REPOSITORY = "https://github.com/FuNz-0/PromptAD.git"
PROMPTAD_COMMIT = "0f86ce0dc1ed59007d51348d8d566aed31360cf9"
PROMPTAD_ROOT = Path("/kaggle/working/PromptAD")

if not PROMPTAD_ROOT.exists():
    subprocess.run(["git", "clone", PROMPTAD_REPOSITORY, str(PROMPTAD_ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROMPTAD_ROOT), "fetch", "origin"], check=True)
subprocess.run(
    ["git", "-C", str(PROMPTAD_ROOT), "checkout", "--detach", PROMPTAD_COMMIT],
    check=True,
)
resolved_commit = subprocess.run(
    ["git", "-C", str(PROMPTAD_ROOT), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if resolved_commit != PROMPTAD_COMMIT:
    raise RuntimeError(f"Wrong PromptAD source commit: {resolved_commit}")
tracked_changes = subprocess.run(
    ["git", "-C", str(PROMPTAD_ROOT), "status", "--porcelain", "--untracked-files=no"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if tracked_changes:
    raise RuntimeError(f"Official tracked source is modified:\n{tracked_changes}")

print(f"Visible CUDA devices: {torch.cuda.device_count()}")
for device_index in range(torch.cuda.device_count()):
    print(f"  GPU {device_index}: {torch.cuda.get_device_name(device_index)}")
print(f"PyTorch: {torch.__version__}; CUDA runtime: {torch.version.cuda}")
# The upstream files import tqdm but never wrap their epoch loops. Create untracked
# observability-only copies so the pinned tracked source remains byte-for-byte clean.
INSTRUMENTED_SCRIPTS = {}
epoch_loop = "for epoch in range(args.Epoch):"
progress_loop = (
    'for epoch in tqdm(range(args.Epoch), '
    'desc=f"{TASK} {args.dataset}/{args.class_name} {args.k_shot}-shot", '
    'unit="epoch", mininterval=1.0, dynamic_ncols=True):'
)
for task_name in ("cls", "seg"):
    source_path = PROMPTAD_ROOT / f"train_{task_name}.py"
    instrumented_path = PROMPTAD_ROOT / f"train_{task_name}_tqdm.py"
    source_text = source_path.read_text(encoding="utf-8")
    if source_text.count(epoch_loop) != 1:
        raise RuntimeError(f"Expected one epoch loop in {source_path}")
    instrumented_path.write_text(
        source_text.replace(epoch_loop, progress_loop), encoding="utf-8"
    )
    INSTRUMENTED_SCRIPTS[task_name] = instrumented_path.name

print(f"Official PromptAD commit: {resolved_commit}")
print("Progress display: overall checkpoint tqdm + per-job 100-epoch tqdm")
print("Dual-GPU routing is validated in the control cell below.")

## Reproduction controls

All optimization parameters below are locked to the official scripts. Normally edit only dataset paths, dataset/task/shot/class selection, and session-resume controls.

The complete suite is much longer than a single free Kaggle session because the official implementation performs full test inference at every epoch. Checkpoints are written after every improvement, so download the produced archive (or publish it as a private Kaggle Dataset), attach it to the next session, and point `RESUME_RESULT_ROOT` at its extracted `result` directory.

In [ ]:
print("====== STEP 2: CONFIGURING THE OFFICIAL TRAINING SUITE ======")

# ------------------------------------------------------------------------------
# USER-CONTROLLABLE SELECTION AND KAGGLE PATHS
# ------------------------------------------------------------------------------
DATASET_NAME = "mvtec"       # "mvtec", "visa", or "both"
SHOTS_TO_TRAIN = [1, 2, 4]
TASKS_TO_TRAIN = ["cls", "seg"]
CLASSES_TO_TRAIN = None       # None = every class; or e.g. ["bottle", "cable"]

MVTEC_ROOT = Path("/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection")
VISA_RAW_ROOT = Path("/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922")
VISA_PREPARED_ROOT = None     # Optional existing .../VisA_pytorch/1cls directory.

RESULT_ROOT = Path("/kaggle/working/promptad_retrained/result")
ARTIFACT_ROOT = Path("/kaggle/working/promptad_retrained")
RESUME_RESULT_ROOT = None     # Optional attached prior session's exact `result` directory.
RETRAIN_EXISTING = False
MAX_JOBS_THIS_SESSION = None  # e.g. 2 = one CLS+SEG class pair; None runs all.
GPU_IDS = [0, 1]             # CLS -> GPU 0, SEG -> GPU 1 by default.
SAVE_BEST_VISUALIZATIONS = False  # No effect on optimization or checkpoints.

# ------------------------------------------------------------------------------
# LOCKED OFFICIAL PAPER/CODE SETTINGS -- DO NOT EDIT FOR REPRODUCTION
# ------------------------------------------------------------------------------
SEED = 111
IMAGE_RESIZE = 240
IMAGE_CROP = 240
METRIC_RESOLUTION = 400
BATCH_SIZE = 400
BACKBONE = "ViT-B-16-plus-240"
PRETRAINED_DATASET = "laion400m_e32"
EPOCHS = 100
LEARNING_RATE = 0.002
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005
ALIGNMENT_WEIGHT = 0.001
NORMAL_CONTEXT_TOKENS = 4
ANOMALY_CONTEXT_TOKENS = 1
LEARNABLE_ANOMALY_SUFFIXES = 4
NORMAL_PROTOTYPES_BY_TASK = {"cls": 3, "seg": 1}

MVTEC_CLASSES = [
    "carpet", "grid", "leather", "tile", "wood", "bottle", "cable",
    "capsule", "hazelnut", "metal_nut", "pill", "screw", "toothbrush",
    "transistor", "zipper",
]
VISA_CLASSES = [
    "candle", "capsules", "cashew", "chewinggum", "fryum", "macaroni1",
    "macaroni2", "pcb1", "pcb2", "pcb3", "pcb4", "pipe_fryum",
]

DATASET_NAME = DATASET_NAME.lower().strip()
if DATASET_NAME not in {"mvtec", "visa", "both"}:
    raise ValueError("DATASET_NAME must be 'mvtec', 'visa', or 'both'.")
DATASETS_TO_TRAIN = ["mvtec", "visa"] if DATASET_NAME == "both" else [DATASET_NAME]
SHOTS_TO_TRAIN = [int(value) for value in SHOTS_TO_TRAIN]
if not SHOTS_TO_TRAIN or set(SHOTS_TO_TRAIN) - {1, 2, 4}:
    raise ValueError("SHOTS_TO_TRAIN must contain values from [1, 2, 4].")
TASKS_TO_TRAIN = [value.lower().strip() for value in TASKS_TO_TRAIN]
if not TASKS_TO_TRAIN or set(TASKS_TO_TRAIN) - {"cls", "seg"}:
    raise ValueError("TASKS_TO_TRAIN must contain 'cls' and/or 'seg'.")
if MAX_JOBS_THIS_SESSION is not None and MAX_JOBS_THIS_SESSION < 1:
    raise ValueError("MAX_JOBS_THIS_SESSION must be None or a positive integer.")
GPU_IDS = [int(value) for value in GPU_IDS]
if not GPU_IDS or len(GPU_IDS) != len(set(GPU_IDS)):
    raise ValueError("GPU_IDS must contain one or more unique CUDA device indices.")
visible_gpu_count = torch.cuda.device_count()
if min(GPU_IDS) < 0 or max(GPU_IDS) >= visible_gpu_count:
    raise RuntimeError(
        f"GPU_IDS={GPU_IDS} but this Kaggle session exposes {visible_gpu_count} GPU(s). "
        "Select the T4 x2 accelerator or use GPU_IDS=[0]."
    )
TASK_GPU_IDS = {
    "cls": GPU_IDS[0],
    "seg": GPU_IDS[1] if len(GPU_IDS) > 1 else GPU_IDS[0],
}

print(f"Datasets: {DATASETS_TO_TRAIN}")
print(f"Shots: {SHOTS_TO_TRAIN}; tasks: {TASKS_TO_TRAIN}")
print(f"Task-to-GPU routing: {TASK_GPU_IDS}")
print("Official optimization configuration is locked.")

## Dataset preparation and official sample selection

MVTec AD is used in its standard directory layout. Raw VisA is converted with PromptAD's own `prepare_visa_public.py` and official `split_csv/1cls.csv`. Symlinks place the mounted data at the hard-coded locations expected by the pinned source without modifying any tracked source file.

PromptAD's checked-in `selected_samples_per_run.txt` files define the 1/2/4 normal references. The preflight records the actual selected filenames in the manifest, which is essential because the upstream loader stores numeric indices rather than filenames.

In [ ]:
import glob

print("====== STEP 3: PREPARING DATASETS AND OFFICIAL REFERENCE SETS ======")

def valid_dataset_root(root, classes, image_suffix):
    root = Path(root)
    return root.is_dir() and all(
        (root / class_name / "train" / "good").is_dir()
        and (root / class_name / "test").is_dir()
        and any((root / class_name / "train" / "good").glob(f"*{image_suffix}"))
        for class_name in classes
    )


def make_exact_symlink(link, target):
    link = Path(link)
    target = Path(target).resolve()
    if not target.is_dir():
        raise FileNotFoundError(f"Dataset directory does not exist: {target}")
    link.parent.mkdir(parents=True, exist_ok=True)
    if link.is_symlink():
        if link.resolve() == target:
            return
        link.unlink()
    elif link.exists():
        raise RuntimeError(f"Refusing to replace non-symlink path: {link}")
    link.symlink_to(target, target_is_directory=True)


dataset_roots = {}
if "mvtec" in DATASETS_TO_TRAIN:
    if not valid_dataset_root(MVTEC_ROOT, MVTEC_CLASSES, ".png"):
        raise FileNotFoundError(
            f"MVTec AD is not in the expected layout at {MVTEC_ROOT}. "
            "Edit MVTEC_ROOT to the directory containing the 15 class folders."
        )
    dataset_roots["mvtec"] = MVTEC_ROOT.resolve()
    make_exact_symlink(
        PROMPTAD_ROOT / "anomaly_detection" / "mvtec_anomaly_detection",
        dataset_roots["mvtec"],
    )

if "visa" in DATASETS_TO_TRAIN:
    prepared_candidate = Path(VISA_PREPARED_ROOT) if VISA_PREPARED_ROOT else None
    if prepared_candidate and valid_dataset_root(prepared_candidate, VISA_CLASSES, ".JPG"):
        prepared_visa = prepared_candidate.resolve()
    else:
        if not (VISA_RAW_ROOT / "split_csv" / "1cls.csv").is_file():
            raise FileNotFoundError(
                f"Raw VisA with split_csv/1cls.csv was not found at {VISA_RAW_ROOT}. "
                "Edit VISA_RAW_ROOT or provide VISA_PREPARED_ROOT."
            )
        # Keep the copied/prepared dataset outside ARTIFACT_ROOT so it is not
        # accidentally included in the checkpoint archive.
        visa_work_root = Path("/kaggle/working/promptad_visa_prepared/VisA_pytorch")
        prepared_visa = visa_work_root / "1cls"
        if not valid_dataset_root(prepared_visa, VISA_CLASSES, ".JPG"):
            subprocess.run(
                [
                    sys.executable,
                    str(PROMPTAD_ROOT / "datasets" / "prepare_visa_public.py"),
                    "--split-type", "1cls",
                    "--data-folder", str(VISA_RAW_ROOT),
                    "--save-folder", str(visa_work_root),
                    "--split-file", str(VISA_RAW_ROOT / "split_csv" / "1cls.csv"),
                ],
                cwd=PROMPTAD_ROOT,
                check=True,
            )
        if not valid_dataset_root(prepared_visa, VISA_CLASSES, ".JPG"):
            raise RuntimeError(f"Official VisA preprocessing did not produce {prepared_visa}")
    dataset_roots["visa"] = prepared_visa.resolve()
    make_exact_symlink(
        PROMPTAD_ROOT / "DATA" / "anomaly_detection" / "VisA_pytorch" / "1cls",
        dataset_roots["visa"],
    )

classes_by_dataset = {"mvtec": MVTEC_CLASSES, "visa": VISA_CLASSES}
suffix_by_dataset = {"mvtec": ".png", "visa": ".JPG"}
seed_dir_by_dataset = {"mvtec": "seeds_mvtec", "visa": "seeds_visa"}

if CLASSES_TO_TRAIN is not None:
    requested_classes = set(CLASSES_TO_TRAIN)
    valid_classes = set().union(*(classes_by_dataset[name] for name in DATASETS_TO_TRAIN))
    unknown_classes = requested_classes - valid_classes
    if unknown_classes:
        raise ValueError(f"Unknown classes for selected datasets: {sorted(unknown_classes)}")
else:
    requested_classes = None


def official_reference_paths(dataset_name, class_name, shot):
    # Mirror the upstream loader's glob and numeric-index selection exactly.
    train_good = dataset_roots[dataset_name] / class_name / "train" / "good"
    suffix = suffix_by_dataset[dataset_name]
    upstream_order = glob.glob(str(train_good / f"*{suffix}"))
    seed_file = (
        PROMPTAD_ROOT / "datasets" / seed_dir_by_dataset[dataset_name]
        / class_name / "selected_samples_per_run.txt"
    )
    prefix = f"#{shot}: "
    matching_lines = [
        line.rstrip("\n") for line in seed_file.read_text(encoding="utf-8").splitlines(keepends=True)
        if line.startswith(prefix)
    ]
    if len(matching_lines) != 1:
        raise RuntimeError(f"Expected one {prefix!r} line in {seed_file}")
    indices = [int(value) for value in matching_lines[0][len(prefix):].split()]
    if len(indices) != shot or any(index >= len(upstream_order) for index in indices):
        raise RuntimeError(
            f"Invalid official {shot}-shot indices for {dataset_name}/{class_name}: "
            f"{indices}; training images={len(upstream_order)}"
        )
    return [str(Path(upstream_order[index]).relative_to(dataset_roots[dataset_name])) for index in indices]


selected_references = {}
jobs = []
for dataset_name in DATASETS_TO_TRAIN:
    selected_classes = [
        class_name for class_name in classes_by_dataset[dataset_name]
        if requested_classes is None or class_name in requested_classes
    ]
    if not selected_classes:
        raise ValueError(f"No selected classes belong to {dataset_name}.")
    for shot in SHOTS_TO_TRAIN:
        for class_name in selected_classes:
            key = f"{dataset_name}/{shot}-shot/{class_name}"
            selected_references[key] = official_reference_paths(dataset_name, class_name, shot)
            for task in TASKS_TO_TRAIN:
                jobs.append({
                    "dataset": dataset_name,
                    "shot": shot,
                    "class_name": class_name,
                    "task": task,
                })

print(f"Dataset preflight passed. Scheduled checkpoints: {len(jobs)}")
print("First official reference selections:")
for key in list(selected_references)[:5]:
    print(f"  {key}: {selected_references[key]}")

## Resumable dual-GPU official launcher

For each class, the notebook launches the official CLS job on GPU 0 and the official SEG job on GPU 1 concurrently. Each subprocess writes to a unique temporary result root, so the upstream scripts never race on their shared CSV. After training, the parent process validates and copies both checkpoints into the canonical result tree and merges only the relevant `i_roc` or `p_roc` cell under a lock.

Each job invokes an untracked, instrumented copy of official `train_cls.py` or `train_seg.py`. The instrumentation only adds `tqdm`; the tracked pinned source, optimization, evaluation, and checkpoint logic remain unchanged. The visual-gallery validation rejects partial, mismatched, or swapped checkpoints before resume or packaging.

In [ ]:
import tempfile
import threading

import pandas as pd

print("====== STEP 4: BUILDING THE RESUMABLE DUAL-GPU LAUNCHER ======")

RESULT_ROOT.mkdir(parents=True, exist_ok=True)
LOG_ROOT = ARTIFACT_ROOT / "logs"
LOG_ROOT.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = ARTIFACT_ROOT / "training_manifest.json"
JOB_WORK_ROOT = Path("/kaggle/working/promptad_parallel_jobs")
JOB_WORK_ROOT.mkdir(parents=True, exist_ok=True)
MERGE_LOCK = threading.Lock()

if RESUME_RESULT_ROOT is not None:
    resume_root = Path(RESUME_RESULT_ROOT)
    if not resume_root.is_dir():
        raise FileNotFoundError(f"RESUME_RESULT_ROOT does not exist: {resume_root}")
    shutil.copytree(resume_root, RESULT_ROOT, dirs_exist_ok=True)
    print(f"Merged prior result tree from {resume_root}")


def checkpoint_path(job, root=RESULT_ROOT):
    task_label = job["task"].upper()
    return (
        Path(root) / job["dataset"] / f"k_{job['shot']}" / "checkpoint"
        / f"{task_label}-Seed_{SEED}-{job['class_name']}-check_point.pt"
    )


def csv_path(job, root=RESULT_ROOT):
    return Path(root) / job["dataset"] / f"k_{job['shot']}" / "csv" / f"Seed_{SEED}-results.csv"


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def validate_checkpoint(path, shot):
    path = Path(path)
    if not path.is_file() or path.stat().st_size < 1024:
        raise RuntimeError(f"Missing or implausibly small checkpoint: {path}")
    state = torch.load(path, map_location="cpu", weights_only=True)
    expected_keys = {"feature_gallery1", "feature_gallery2", "text_features"}
    if set(state) != expected_keys:
        raise RuntimeError(f"Unexpected keys in {path}: {sorted(state)}")
    expected_patches = shot * 15 * 15
    for key in ("feature_gallery1", "feature_gallery2"):
        if state[key].ndim != 2 or state[key].shape[0] != expected_patches:
            raise RuntimeError(
                f"{path}: {key} has shape {tuple(state[key].shape)}, "
                f"expected first dimension {expected_patches}."
            )
    if tuple(state["text_features"].shape) != (2, 640):
        raise RuntimeError(
            f"{path}: text_features has shape {tuple(state['text_features'].shape)}, expected (2, 640)."
        )
    try:
        display_path = str(path.relative_to(ARTIFACT_ROOT))
    except ValueError:
        display_path = str(path)
    return {
        "path": display_path,
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
        "shapes": {key: list(value.shape) for key, value in state.items()},
    }


def official_command(job, gpu_id, job_result_root):
    script = INSTRUMENTED_SCRIPTS[job["task"]]
    return [
        sys.executable, script,
        "--dataset", job["dataset"],
        "--class_name", job["class_name"],
        "--k-shot", str(job["shot"]),
        "--img-resize", str(IMAGE_RESIZE),
        "--img-cropsize", str(IMAGE_CROP),
        "--resolution", str(METRIC_RESOLUTION),
        "--batch-size", str(BATCH_SIZE),
        "--vis", str(SAVE_BEST_VISUALIZATIONS).lower(),
        "--root-dir", str(job_result_root),
        "--seed", str(SEED),
        "--gpu-id", str(gpu_id),
        "--backbone", BACKBONE,
        "--pretrained_dataset", PRETRAINED_DATASET,
        "--n_ctx", str(NORMAL_CONTEXT_TOKENS),
        "--n_ctx_ab", str(ANOMALY_CONTEXT_TOKENS),
        "--n_pro", str(NORMAL_PROTOTYPES_BY_TASK[job["task"]]),
        "--n_pro_ab", str(LEARNABLE_ANOMALY_SUFFIXES),
        "--Epoch", str(EPOCHS),
        "--lr", str(LEARNING_RATE),
        "--momentum", str(MOMENTUM),
        "--weight_decay", str(WEIGHT_DECAY),
        "--lambda1", str(ALIGNMENT_WEIGHT),
    ]


def merge_official_csv(job, job_result_root):
    source_path = csv_path(job, job_result_root)
    if not source_path.is_file():
        raise RuntimeError(f"Official job did not produce its metric CSV: {source_path}")
    source = pd.read_csv(source_path, index_col=0)
    row_name = f"{job['dataset']}-{job['class_name']}"
    metric_column = "i_roc" if job["task"] == "cls" else "p_roc"
    if row_name not in source.index or metric_column not in source.columns:
        raise RuntimeError(f"Missing {row_name}/{metric_column} in {source_path}")

    destination_path = csv_path(job)
    destination_path.parent.mkdir(parents=True, exist_ok=True)
    if destination_path.is_file():
        destination = pd.read_csv(destination_path, index_col=0)
    else:
        destination = source.copy()
        for column in ("i_roc", "p_roc"):
            destination[column] = 0.0
    destination.loc[row_name, metric_column] = source.loc[row_name, metric_column]
    temporary_path = destination_path.with_suffix(".tmp.csv")
    destination.to_csv(temporary_path, header=True, float_format="%.2f")
    os.replace(temporary_path, destination_path)


# Download and checksum the official backbone once before two subprocesses
# can attempt to access the same cache file.
prewarm_code = (
    "from PromptAD.CLIPAD.pretrained import get_pretrained_cfg, download_pretrained; "
    f"cfg=get_pretrained_cfg({BACKBONE!r}, {PRETRAINED_DATASET!r}); "
    "path=download_pretrained(cfg); print(f'Backbone cache ready: {path}')"
)
subprocess.run([sys.executable, "-c", prewarm_code], cwd=PROMPTAD_ROOT, check=True)


package_versions = {}
for distribution in (
    "torch", "torchvision", "open-clip-torch", "timm", "numpy", "scipy",
    "scikit-image", "scikit-learn", "opencv-python-headless",
):
    try:
        package_versions[distribution] = importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        package_versions[distribution] = None

manifest = {
    "schema_version": 2,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "official_repository": PROMPTAD_REPOSITORY,
    "official_commit": PROMPTAD_COMMIT,
    "paper": "PromptAD: Learning Prompts with only Normal Samples for Few-Shot Anomaly Detection (CVPR 2024)",
    "paper_url": "https://arxiv.org/abs/2404.05231",
    "checkpoint_semantics": "Official inference buffers only; frozen CLIP backbone is downloaded separately.",
    "checkpoint_selection": "Best test AUROC over 100 epochs, matching official train scripts.",
    "progress_instrumentation": "Untracked script copies wrap only the existing epoch range in tqdm.",
    "parallelism": {
        "strategy": "Concurrent class-matched CLS and SEG subprocesses with isolated result roots.",
        "task_gpu_ids": TASK_GPU_IDS,
        "merge": "Parent-locked checkpoint and metric merge after validation.",
    },
    "settings": {
        "seed": SEED,
        "image_resize": IMAGE_RESIZE,
        "image_crop": IMAGE_CROP,
        "metric_resolution": METRIC_RESOLUTION,
        "batch_size": BATCH_SIZE,
        "backbone": BACKBONE,
        "pretrained_dataset": PRETRAINED_DATASET,
        "epochs": EPOCHS,
        "optimizer": "SGD",
        "learning_rate": LEARNING_RATE,
        "momentum": MOMENTUM,
        "weight_decay": WEIGHT_DECAY,
        "scheduler": "CosineAnnealingLR(T_max=100, eta_min=1e-5)",
        "alignment_weight": ALIGNMENT_WEIGHT,
        "normal_context_tokens": NORMAL_CONTEXT_TOKENS,
        "anomaly_context_tokens": ANOMALY_CONTEXT_TOKENS,
        "learnable_anomaly_suffixes": LEARNABLE_ANOMALY_SUFFIXES,
        "normal_prototypes_by_task": NORMAL_PROTOTYPES_BY_TASK,
        "precision": "fp16",
    },
    "dataset_roots": {key: str(value) for key, value in dataset_roots.items()},
    "selected_normal_references": selected_references,
    "runtime": {
        "python": sys.version,
        "platform": platform.platform(),
        "gpus": {str(index): torch.cuda.get_device_name(index) for index in GPU_IDS},
        "cuda_runtime": torch.version.cuda,
        "packages": package_versions,
    },
    "jobs": [],
}


def write_manifest():
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")


def run_official_job(job, gpu_id):
    central_path = checkpoint_path(job)
    if central_path.exists() and not RETRAIN_EXISTING:
        metadata = validate_checkpoint(central_path, job["shot"])
        print(f"SKIP valid checkpoint: {metadata['path']}")
        return {**job, "gpu_id": gpu_id, "status": "reused", "checkpoint": metadata}

    prefix = f"{job['dataset']}_{job['shot']}shot_{job['class_name']}_{job['task']}_"
    job_result_root = Path(tempfile.mkdtemp(prefix=prefix, dir=JOB_WORK_ROOT))
    command = official_command(job, gpu_id, job_result_root)
    log_path = LOG_ROOT / f"{prefix.rstrip('_')}.log"
    label = f"GPU {gpu_id} {job['task'].upper()} {job['dataset']}/{job['class_name']}"
    print("\n" + "=" * 88)
    print(f"START {label} ({job['shot']}-shot)")
    print(" ".join(command))
    started = time.time()
    environment = os.environ.copy()
    environment["PYTHONUNBUFFERED"] = "1"
    environment["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
    with log_path.open("w", encoding="utf-8") as log_handle:
        process = subprocess.Popen(
            command,
            cwd=PROMPTAD_ROOT,
            env=environment,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in process.stdout:
            print(f"[{label}] {line}", end="")
            log_handle.write(line)
        return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(
            f"{label} failed with code {return_code}; see {log_path}. "
            f"Isolated outputs retained at {job_result_root}."
        )

    isolated_checkpoint = checkpoint_path(job, job_result_root)
    validate_checkpoint(isolated_checkpoint, job["shot"])
    central_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(isolated_checkpoint, central_path)
    with MERGE_LOCK:
        merge_official_csv(job, job_result_root)
    metadata = validate_checkpoint(central_path, job["shot"])
    shutil.rmtree(job_result_root)
    return {
        **job,
        "gpu_id": gpu_id,
        "status": "trained",
        "elapsed_seconds": round(time.time() - started, 3),
        "command": command,
        "log": str(log_path.relative_to(ARTIFACT_ROOT)),
        "checkpoint": metadata,
    }

write_manifest()
print(f"Dual-GPU launcher ready for {len(jobs)} checkpoints.")
print(f"Routing: CLS -> GPU {TASK_GPU_IDS['cls']}; SEG -> GPU {TASK_GPU_IDS['seg']}")

## Train two task checkpoints concurrently

Each overall progress step represents one checkpoint; a completed dual-GPU class pair advances it twice. When both tasks are selected, GPU 0 trains CLS while GPU 1 trains SEG. `MAX_JOBS_THIS_SESSION=2` therefore completes one class pair; `None` processes every selected class pair. The manifest is updated after every successful pair, and valid resumed checkpoints are skipped.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

print("====== STEP 5: DUAL-GPU PROMPTAD TRAINING ======")

completed_paths = {
    entry.get("checkpoint", {}).get("path") for entry in manifest["jobs"]
    if entry.get("checkpoint")
}
launched = 0
grouped_jobs = {}
for job in jobs:
    group_key = (job["dataset"], job["shot"], job["class_name"])
    grouped_jobs.setdefault(group_key, []).append(job)

with tqdm(
    total=len(jobs), desc="PromptAD checkpoints", unit="checkpoint", dynamic_ncols=True
) as checkpoint_progress:
    for group_key, class_jobs in grouped_jobs.items():
        jobs_to_train = []
        for job in class_jobs:
            expected_relative = str(checkpoint_path(job).relative_to(ARTIFACT_ROOT))
            if expected_relative in completed_paths:
                checkpoint_progress.update(1)
                continue
            if checkpoint_path(job).is_file() and not RETRAIN_EXISTING:
                result = run_official_job(job, TASK_GPU_IDS[job["task"]])
                manifest["jobs"].append(result)
                completed_paths.add(result["checkpoint"]["path"])
                checkpoint_progress.update(1)
                write_manifest()
                continue
            jobs_to_train.append(job)

        if MAX_JOBS_THIS_SESSION is not None:
            remaining_budget = MAX_JOBS_THIS_SESSION - launched
            if remaining_budget <= 0:
                print(f"Session job limit reached ({MAX_JOBS_THIS_SESSION}).")
                break
            jobs_to_train = jobs_to_train[:remaining_budget]

        if not jobs_to_train:
            continue

        worker_count = min(
            len(jobs_to_train),
            len({TASK_GPU_IDS[job["task"]] for job in jobs_to_train}),
        )
        with ThreadPoolExecutor(max_workers=worker_count) as executor:
            futures = {
                executor.submit(
                    run_official_job, job, TASK_GPU_IDS[job["task"]]
                ): job
                for job in jobs_to_train
            }
            pair_results = []
            for future in as_completed(futures):
                pair_results.append(future.result())

        for result in sorted(pair_results, key=lambda item: item["task"]):
            manifest["jobs"].append(result)
            completed_paths.add(result["checkpoint"]["path"])
            launched += int(result["status"] == "trained")
            checkpoint_progress.update(1)
        write_manifest()

complete_count = len(completed_paths)
print(f"Valid checkpoints in this artifact: {complete_count}/{len(jobs)} selected jobs")
if complete_count < len(jobs):
    print("This is a valid partial shard. Package it below and resume in another Kaggle session.")
else:
    print("Selected dual-GPU PromptAD training suite is complete.")

## Validate and package

The archive preserves the official `result/<dataset>/k_<shot>/checkpoint/...` paths and includes SHA-256 hashes, exact selected reference filenames, runtime versions, commands, CSVs, and logs. A future few-shot wrapper should load both the class-matched `CLS` and `SEG` files with `strict=False`, exactly as the official test scripts do, while obtaining the frozen `ViT-B-16-plus-240 / laion400m_e32` backbone separately.

In [ ]:
import zipfile

print("====== STEP 6: VALIDATING AND PACKAGING ARTIFACTS ======")

checkpoint_index = {}
for job in jobs:
    path = checkpoint_path(job)
    if not path.is_file():
        continue
    metadata = validate_checkpoint(path, job["shot"])
    key = f"{job['dataset']}/{job['shot']}-shot/{job['class_name']}/{job['task']}"
    checkpoint_index[key] = metadata

INDEX_PATH = ARTIFACT_ROOT / "checkpoint_index.json"
INDEX_PATH.write_text(json.dumps(checkpoint_index, indent=2), encoding="utf-8")
readme_text = (
    "PromptAD paper-faithful retrained checkpoints\n\n"
    f"Official source: {PROMPTAD_REPOSITORY}\n"
    f"Pinned commit: {PROMPTAD_COMMIT}\n"
    "Paper: https://arxiv.org/abs/2404.05231\n\n"
    f"This artifact contains {len(checkpoint_index)} of {len(jobs)} checkpoints "
    "selected in the notebook.\n"
    "Paths retain the official result/<dataset>/k_<shot>/checkpoint convention.\n"
    "Each class needs a CLS checkpoint for image scoring and a SEG checkpoint for pixel maps.\n"
    "The checkpoint files contain feature_gallery1, feature_gallery2, and text_features;\n"
    "the frozen LAION-400M ViT-B/16+ backbone is not duplicated in every file.\n\n"
    "See training_manifest.json for exact settings, selected normal references, commands,\n"
    "runtime versions, checkpoint-selection behavior, hashes, and completion state.\n"
)
(ARTIFACT_ROOT / "README.txt").write_text(readme_text, encoding="utf-8")

archive_path = Path("/kaggle/working/promptad_retrained_checkpoints.zip")
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(ARTIFACT_ROOT.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(ARTIFACT_ROOT.parent))

if not zipfile.is_zipfile(archive_path):
    raise RuntimeError(f"Invalid artifact archive: {archive_path}")
with zipfile.ZipFile(archive_path) as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise RuntimeError(f"Corrupt archive member: {bad_member}")

print(f"Checkpoint index entries: {len(checkpoint_index)}")
print(f"Archive: {archive_path} ({archive_path.stat().st_size / 1024**2:.1f} MiB)")
print("Download the ZIP from Kaggle's Output panel.")